# ASL Sign Recognition - Multi-Stream TCN-Transformer

Key innovations:
1. **Multi-Stream Architecture**: Separate encoders for hands vs face+pose
2. **TCN + Transformer Hybrid**: Dilated convolutions + attention
3. **Enhanced Features**: Position, velocity, acceleration, hand distances
4. **Advanced Augmentation**: Temporal scaling, hand mirroring, mixup
5. **Better Training**: Focal loss, cosine annealing, larger capacity

In [ ]:
import json
import os
import math
from typing import List, Tuple, Optional, Dict

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from torch.utils.data import Dataset, DataLoader, Sampler

from torch.amp.grad_scaler import GradScaler
from torch.amp.autocast_mode import autocast

import warnings
warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = device.type == "cuda"
print(f"Using device: {device}, AMP: {use_amp}")

In [ ]:
# Configuration
BASE_PATH = r"/kaggle/input/asl-signs"
TRAIN_FILE = r"/kaggle/input/asl-signs/train.csv"
SIGN_INDEX_FILE = r"/kaggle/input/asl-signs/sign_to_prediction_index_map.json"

with open(SIGN_INDEX_FILE, "r") as f:
    SIGN2INDEX = json.load(f)

NUM_CLASSES = len(SIGN2INDEX)
print(f"Number of classes: {NUM_CLASSES}")

In [ ]:
# Landmark configuration
# We'll process: left_hand (21), right_hand (21), pose (33), selected face (134)

FACE_LANDMARK_INDICES = {
    'nose': [1, 2, 4, 5, 6, 19, 94, 168, 197, 195],
    'left_eye': [33, 133, 160, 159, 158, 157, 173, 144, 145, 153, 154, 155, 156, 246, 7, 163],
    'right_eye': [263, 362, 387, 386, 385, 384, 398, 373, 374, 380, 381, 382, 466, 388, 390, 249],
    'left_eyebrow': [70, 63, 105, 66, 107, 55, 65, 52],
    'right_eyebrow': [300, 293, 334, 296, 336, 285, 295, 282],
    'mouth_outer': [61, 146, 91, 181, 84, 17, 314, 405, 321, 375, 291, 409, 270, 269, 267, 0, 37, 39, 40, 185],
    'mouth_inner': [78, 191, 80, 81, 82, 13, 312, 311, 310, 415, 308, 324, 318, 402, 317, 14, 87, 178, 88, 95],
    'face_oval': [10, 338, 297, 332, 284, 251, 389, 356, 454, 323, 361, 288, 397, 365, 379, 378, 400, 377, 
                  152, 148, 176, 149, 150, 136, 172, 58, 132, 93, 234, 127, 162, 21, 54, 103, 67, 109]
}

SELECTED_FACE_INDICES = sorted(set(idx for indices in FACE_LANDMARK_INDICES.values() for idx in indices))

# Landmark counts
N_LEFT_HAND = 21
N_RIGHT_HAND = 21
N_POSE = 33
N_FACE = len(SELECTED_FACE_INDICES)

print(f"Landmarks: Left Hand={N_LEFT_HAND}, Right Hand={N_RIGHT_HAND}, Pose={N_POSE}, Face={N_FACE}")
print(f"Total landmarks: {N_LEFT_HAND + N_RIGHT_HAND + N_POSE + N_FACE}")

In [ ]:
def load_parquet_data(file_path: str) -> Dict[str, np.ndarray]:
    """
    Load and preprocess parquet file into separate arrays for each body part.
    Returns dict with keys: 'left_hand', 'right_hand', 'pose', 'face'
    Each array has shape (T, N_landmarks, 2) for x,y coordinates
    """
    df = pd.read_parquet(os.path.join(BASE_PATH, file_path))
    
    frames = sorted(df['frame'].unique())
    n_frames = len(frames)
    frame_to_idx = {f: i for i, f in enumerate(frames)}
    
    # Initialize arrays
    left_hand = np.zeros((n_frames, N_LEFT_HAND, 2), dtype=np.float32)
    right_hand = np.zeros((n_frames, N_RIGHT_HAND, 2), dtype=np.float32)
    pose = np.zeros((n_frames, N_POSE, 2), dtype=np.float32)
    face = np.zeros((n_frames, N_FACE, 2), dtype=np.float32)
    
    # Face index mapping
    face_idx_map = {orig: new for new, orig in enumerate(SELECTED_FACE_INDICES)}
    
    # Get nose position for normalization (pose landmark 0)
    nose_df = df[(df['type'] == 'pose') & (df['landmark_index'] == 0)]
    nose_positions = {}
    for _, row in nose_df.iterrows():
        nose_positions[row['frame']] = np.array([row['x'], row['y']], dtype=np.float32)
    
    # Fill arrays
    for _, row in df.iterrows():
        frame_idx = frame_to_idx[row['frame']]
        lm_idx = int(row['landmark_index'])
        coords = np.array([row['x'], row['y']], dtype=np.float32)
        
        # Normalize by nose position
        if row['frame'] in nose_positions:
            coords = coords - nose_positions[row['frame']]
        
        lm_type = row['type']
        if lm_type == 'left_hand' and lm_idx < N_LEFT_HAND:
            left_hand[frame_idx, lm_idx] = coords
        elif lm_type == 'right_hand' and lm_idx < N_RIGHT_HAND:
            right_hand[frame_idx, lm_idx] = coords
        elif lm_type == 'pose' and lm_idx < N_POSE:
            pose[frame_idx, lm_idx] = coords
        elif lm_type == 'face' and lm_idx in face_idx_map:
            face[frame_idx, face_idx_map[lm_idx]] = coords
    
    # Interpolate missing values
    for arr in [left_hand, right_hand, pose, face]:
        for lm in range(arr.shape[1]):
            for c in range(2):
                col = arr[:, lm, c]
                if np.any(col != 0):
                    # Forward fill then backward fill
                    mask = col != 0
                    if not mask.all():
                        indices = np.arange(len(col))
                        col_filled = np.interp(indices, indices[mask], col[mask])
                        arr[:, lm, c] = col_filled
    
    return {
        'left_hand': left_hand,
        'right_hand': right_hand,
        'pose': pose,
        'face': face
    }

In [ ]:
def compute_hand_features(hand: np.ndarray) -> np.ndarray:
    """
    Compute additional features for hand landmarks.
    Input: (T, 21, 2)
    Output: (T, feature_dim)
    
    Features:
    - Flattened coordinates (21 * 2 = 42)
    - Velocity (21 * 2 = 42)
    - Key distances: thumb-index, index-middle, thumb-pinky, palm width (4)
    - Finger curl angles (5 fingers)
    Total: 42 + 42 + 4 + 5 = 93
    """
    T = hand.shape[0]
    
    # Flatten coordinates
    flat = hand.reshape(T, -1)  # (T, 42)
    
    # Velocity
    vel = np.zeros_like(flat)
    vel[1:] = flat[1:] - flat[:-1]
    
    # Key distances
    # Hand landmark indices: 0=wrist, 4=thumb_tip, 8=index_tip, 12=middle_tip, 16=ring_tip, 20=pinky_tip
    thumb_index = np.linalg.norm(hand[:, 4] - hand[:, 8], axis=1, keepdims=True)
    index_middle = np.linalg.norm(hand[:, 8] - hand[:, 12], axis=1, keepdims=True)
    thumb_pinky = np.linalg.norm(hand[:, 4] - hand[:, 20], axis=1, keepdims=True)
    palm_width = np.linalg.norm(hand[:, 5] - hand[:, 17], axis=1, keepdims=True)  # index_mcp to pinky_mcp
    
    distances = np.concatenate([thumb_index, index_middle, thumb_pinky, palm_width], axis=1)  # (T, 4)
    
    # Finger curl angles (angle at PIP joint for each finger)
    # Thumb: 2-3-4, Index: 5-6-7, Middle: 9-10-11, Ring: 13-14-15, Pinky: 17-18-19
    def compute_angle(p1, p2, p3):
        """Compute angle at p2 formed by p1-p2-p3"""
        v1 = p1 - p2
        v2 = p3 - p2
        cos_angle = np.sum(v1 * v2, axis=1) / (np.linalg.norm(v1, axis=1) * np.linalg.norm(v2, axis=1) + 1e-8)
        return np.arccos(np.clip(cos_angle, -1, 1))
    
    finger_angles = np.stack([
        compute_angle(hand[:, 2], hand[:, 3], hand[:, 4]),    # thumb
        compute_angle(hand[:, 5], hand[:, 6], hand[:, 7]),    # index
        compute_angle(hand[:, 9], hand[:, 10], hand[:, 11]),  # middle
        compute_angle(hand[:, 13], hand[:, 14], hand[:, 15]), # ring
        compute_angle(hand[:, 17], hand[:, 18], hand[:, 19])  # pinky
    ], axis=1)  # (T, 5)
    
    # Replace NaN with 0
    finger_angles = np.nan_to_num(finger_angles, 0)
    
    return np.concatenate([flat, vel, distances, finger_angles], axis=1)  # (T, 93)

In [ ]:
def compute_pose_face_features(pose: np.ndarray, face: np.ndarray) -> np.ndarray:
    """
    Compute features for pose and face landmarks.
    Input: pose (T, 33, 2), face (T, N_FACE, 2)
    Output: (T, feature_dim)
    
    Features:
    - Flattened pose coordinates (33 * 2 = 66)
    - Pose velocity (66)
    - Flattened face coordinates (N_FACE * 2)
    - Face velocity (N_FACE * 2)
    - Key pose distances: shoulder width, arm lengths (4)
    """
    T = pose.shape[0]
    
    # Pose features
    pose_flat = pose.reshape(T, -1)
    pose_vel = np.zeros_like(pose_flat)
    pose_vel[1:] = pose_flat[1:] - pose_flat[:-1]
    
    # Face features
    face_flat = face.reshape(T, -1)
    face_vel = np.zeros_like(face_flat)
    face_vel[1:] = face_flat[1:] - face_flat[:-1]
    
    # Key pose distances
    # Pose indices: 11=left_shoulder, 12=right_shoulder, 13=left_elbow, 14=right_elbow, 15=left_wrist, 16=right_wrist
    shoulder_width = np.linalg.norm(pose[:, 11] - pose[:, 12], axis=1, keepdims=True)
    left_arm = np.linalg.norm(pose[:, 11] - pose[:, 15], axis=1, keepdims=True)
    right_arm = np.linalg.norm(pose[:, 12] - pose[:, 16], axis=1, keepdims=True)
    hand_distance = np.linalg.norm(pose[:, 15] - pose[:, 16], axis=1, keepdims=True)
    
    pose_distances = np.concatenate([shoulder_width, left_arm, right_arm, hand_distance], axis=1)
    
    return np.concatenate([pose_flat, pose_vel, face_flat, face_vel, pose_distances], axis=1)

In [ ]:
def process_video(file_path: str) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Process a video file and return features for each stream.
    Returns: (left_hand_features, right_hand_features, pose_face_features)
    """
    data = load_parquet_data(file_path)
    
    left_features = compute_hand_features(data['left_hand'])    # (T, 93)
    right_features = compute_hand_features(data['right_hand'])  # (T, 93)
    pose_face_features = compute_pose_face_features(data['pose'], data['face'])  # (T, ~400)
    
    return left_features, right_features, pose_face_features

In [ ]:
# Data augmentation functions

def temporal_scale(features: np.ndarray, scale_range: Tuple[float, float] = (0.8, 1.2)) -> np.ndarray:
    """Randomly scale temporal dimension (speed up or slow down)"""
    scale = np.random.uniform(*scale_range)
    T = features.shape[0]
    new_T = int(T * scale)
    if new_T < 2:
        return features
    
    indices = np.linspace(0, T - 1, new_T).astype(int)
    return features[indices]


def add_noise(features: np.ndarray, noise_std: float = 0.003) -> np.ndarray:
    """Add Gaussian noise"""
    noise = np.random.normal(0, noise_std, features.shape)
    return features + noise


def temporal_crop(features: np.ndarray, max_crop_ratio: float = 0.1) -> np.ndarray:
    """Randomly crop start/end of sequence"""
    T = features.shape[0]
    if T < 10:
        return features
    
    start_crop = int(np.random.uniform(0, max_crop_ratio) * T)
    end_crop = int(np.random.uniform(0, max_crop_ratio) * T)
    
    return features[start_crop:T - end_crop] if T - end_crop > start_crop else features


def spatial_shift(features: np.ndarray, shift_std: float = 0.02) -> np.ndarray:
    """Add random spatial shift"""
    shift = np.random.normal(0, shift_std, (1, features.shape[1]))
    return features + shift

In [ ]:
class ASLMultiStreamDataset(Dataset):
    def __init__(self, video_data: List[Tuple], labels: np.ndarray, 
                 max_frames: int = 128, augment: bool = False):
        """
        video_data: List of (left_hand, right_hand, pose_face) tuples
        """
        self.video_data = video_data
        self.labels = labels
        self.max_frames = max_frames
        self.augment = augment
    
    def __len__(self):
        return len(self.video_data)
    
    def __getitem__(self, idx):
        left, right, pose_face = self.video_data[idx]
        label = self.labels[idx]
        
        # Apply augmentation to all streams consistently
        if self.augment:
            # Temporal augmentation (same for all streams)
            if np.random.random() > 0.5:
                scale = np.random.uniform(0.8, 1.2)
                T = left.shape[0]
                new_T = max(2, int(T * scale))
                indices = np.linspace(0, T - 1, new_T).astype(int)
                left, right, pose_face = left[indices], right[indices], pose_face[indices]
            
            if np.random.random() > 0.5:
                left = add_noise(left)
                right = add_noise(right)
                pose_face = add_noise(pose_face, noise_std=0.002)
            
            if np.random.random() > 0.5 and left.shape[0] > 20:
                T = left.shape[0]
                start = np.random.randint(0, max(1, T // 10))
                end = T - np.random.randint(0, max(1, T // 10))
                if end > start:
                    left, right, pose_face = left[start:end], right[start:end], pose_face[start:end]
            
            # Random hand swap (data augmentation for sign symmetry)
            if np.random.random() > 0.7:
                left, right = right.copy(), left.copy()
        
        # Subsample if too long
        T = left.shape[0]
        if T > self.max_frames:
            indices = np.linspace(0, T - 1, self.max_frames).astype(int)
            left, right, pose_face = left[indices], right[indices], pose_face[indices]
        
        return (
            torch.tensor(left, dtype=torch.float32),
            torch.tensor(right, dtype=torch.float32),
            torch.tensor(pose_face, dtype=torch.float32),
            label
        )

In [ ]:
def collate_multistream(batch):
    """Collate function for multi-stream data"""
    lefts, rights, pose_faces, labels = zip(*batch)
    
    B = len(lefts)
    lengths = [l.shape[0] for l in lefts]
    max_len = max(lengths)
    
    left_dim = lefts[0].shape[1]
    right_dim = rights[0].shape[1]
    pf_dim = pose_faces[0].shape[1]
    
    left_padded = torch.zeros(B, max_len, left_dim)
    right_padded = torch.zeros(B, max_len, right_dim)
    pf_padded = torch.zeros(B, max_len, pf_dim)
    mask = torch.zeros(B, max_len, dtype=torch.bool)
    
    for i, (l, r, pf) in enumerate(zip(lefts, rights, pose_faces)):
        T = l.shape[0]
        left_padded[i, :T] = l
        right_padded[i, :T] = r
        pf_padded[i, :T] = pf
        mask[i, :T] = 1
    
    return left_padded, right_padded, pf_padded, mask, torch.tensor(labels)

In [ ]:
class BucketBatchSampler(Sampler):
    """Groups sequences by length for efficient batching"""
    def __init__(self, lengths, batch_size, drop_last=False):
        self.lengths = lengths
        self.batch_size = batch_size
        self.drop_last = drop_last

    def __iter__(self):
        sorted_idxs = np.argsort(self.lengths)
        buckets = []
        for i in range(0, len(sorted_idxs), self.batch_size):
            bucket = sorted_idxs[i:i + self.batch_size]
            if len(bucket) == self.batch_size or not self.drop_last:
                buckets.append(bucket)
        np.random.shuffle(buckets)
        for bucket in buckets:
            yield list(bucket)

    def __len__(self):
        if self.drop_last:
            return len(self.lengths) // self.batch_size
        return (len(self.lengths) + self.batch_size - 1) // self.batch_size

In [ ]:
# ============== MODEL ARCHITECTURE ==============

class TemporalConvBlock(nn.Module):
    """Temporal Convolutional Block with dilated convolutions"""
    def __init__(self, in_channels, out_channels, kernel_size=3, dilation=1, dropout=0.1):
        super().__init__()
        padding = (kernel_size - 1) * dilation // 2
        
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size, 
                               padding=padding, dilation=dilation)
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size,
                               padding=padding, dilation=dilation)
        self.norm1 = nn.BatchNorm1d(out_channels)
        self.norm2 = nn.BatchNorm1d(out_channels)
        self.dropout = nn.Dropout(dropout)
        
        self.residual = nn.Conv1d(in_channels, out_channels, 1) if in_channels != out_channels else nn.Identity()
    
    def forward(self, x):
        # x: (B, C, T)
        residual = self.residual(x)
        
        out = self.conv1(x)
        out = self.norm1(out)
        out = F.gelu(out)
        out = self.dropout(out)
        
        out = self.conv2(out)
        out = self.norm2(out)
        out = F.gelu(out + residual)
        
        return out


class TCNEncoder(nn.Module):
    """Temporal Convolutional Network with increasing dilation"""
    def __init__(self, input_dim, hidden_dim=128, num_layers=4, dropout=0.1):
        super().__init__()
        
        self.input_proj = nn.Linear(input_dim, hidden_dim)
        
        self.layers = nn.ModuleList()
        for i in range(num_layers):
            dilation = 2 ** i  # 1, 2, 4, 8
            self.layers.append(TemporalConvBlock(hidden_dim, hidden_dim, dilation=dilation, dropout=dropout))
    
    def forward(self, x):
        # x: (B, T, D)
        x = self.input_proj(x)  # (B, T, H)
        x = x.transpose(1, 2)   # (B, H, T)
        
        for layer in self.layers:
            x = layer(x)
        
        return x.transpose(1, 2)  # (B, T, H)

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        x = x + self.pe[:x.size(1)]
        return self.dropout(x)


class TransformerEncoder(nn.Module):
    """Transformer encoder with pre-norm"""
    def __init__(self, d_model, n_heads=4, n_layers=2, dropout=0.1):
        super().__init__()
        
        self.pos_enc = PositionalEncoding(d_model, dropout=dropout)
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            activation='gelu',
            batch_first=True,
            norm_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, n_layers)
        self.norm = nn.LayerNorm(d_model)
    
    def forward(self, x, mask):
        # x: (B, T, D), mask: (B, T)
        B = x.size(0)
        
        x = self.pos_enc(x)
        
        # Add CLS token
        cls = self.cls_token.expand(B, 1, -1)
        x = torch.cat([cls, x], dim=1)
        
        # Update mask for CLS
        cls_mask = torch.ones(B, 1, device=mask.device, dtype=torch.bool)
        mask = torch.cat([cls_mask, mask], dim=1)
        
        x = self.encoder(x, src_key_padding_mask=~mask)
        x = self.norm(x)
        
        return x[:, 0]  # Return CLS token

In [ ]:
class StreamEncoder(nn.Module):
    """Encoder for a single stream: TCN -> Transformer"""
    def __init__(self, input_dim, hidden_dim=128, tcn_layers=3, transformer_layers=2, 
                 n_heads=4, dropout=0.1):
        super().__init__()
        
        self.tcn = TCNEncoder(input_dim, hidden_dim, tcn_layers, dropout)
        self.transformer = TransformerEncoder(hidden_dim, n_heads, transformer_layers, dropout)
    
    def forward(self, x, mask):
        # x: (B, T, D)
        x = self.tcn(x)  # (B, T, H)
        return self.transformer(x, mask)  # (B, H)

In [ ]:
class CrossAttentionFusion(nn.Module):
    """Cross-attention to fuse multiple stream embeddings"""
    def __init__(self, d_model, n_heads=4, dropout=0.1):
        super().__init__()
        
        self.query_proj = nn.Linear(d_model, d_model)
        self.key_proj = nn.Linear(d_model, d_model)
        self.value_proj = nn.Linear(d_model, d_model)
        
        self.attention = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, query, key_value):
        # query: (B, D), key_value: (B, N, D) where N is number of other streams
        query = query.unsqueeze(1)  # (B, 1, D)
        
        q = self.query_proj(query)
        k = self.key_proj(key_value)
        v = self.value_proj(key_value)
        
        attended, _ = self.attention(q, k, v)
        out = self.norm(query + self.dropout(attended))
        
        return out.squeeze(1)  # (B, D)

In [ ]:
class MultiStreamASLModel(nn.Module):
    """
    Multi-stream model with:
    - Separate encoders for left hand, right hand, and pose+face
    - Cross-attention fusion
    - Classification head
    """
    def __init__(self, left_dim, right_dim, pose_face_dim, num_classes,
                 hidden_dim=192, tcn_layers=4, transformer_layers=3, 
                 n_heads=6, dropout=0.15):
        super().__init__()
        
        # Stream encoders
        self.left_encoder = StreamEncoder(
            left_dim, hidden_dim, tcn_layers, transformer_layers, n_heads, dropout
        )
        self.right_encoder = StreamEncoder(
            right_dim, hidden_dim, tcn_layers, transformer_layers, n_heads, dropout
        )
        self.pose_face_encoder = StreamEncoder(
            pose_face_dim, hidden_dim, tcn_layers, transformer_layers, n_heads, dropout
        )
        
        # Cross-attention fusion for each stream
        self.left_fusion = CrossAttentionFusion(hidden_dim, n_heads, dropout)
        self.right_fusion = CrossAttentionFusion(hidden_dim, n_heads, dropout)
        self.pose_face_fusion = CrossAttentionFusion(hidden_dim, n_heads, dropout)
        
        # Final fusion and classification
        self.final_norm = nn.LayerNorm(hidden_dim * 3)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 3, hidden_dim * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, num_classes)
        )
        
        # Initialize weights
        self._init_weights()
    
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
    
    def forward(self, left, right, pose_face, mask):
        # Encode each stream
        left_emb = self.left_encoder(left, mask)         # (B, H)
        right_emb = self.right_encoder(right, mask)      # (B, H)
        pose_face_emb = self.pose_face_encoder(pose_face, mask)  # (B, H)
        
        # Cross-attention fusion
        # Each stream attends to the other two
        other_for_left = torch.stack([right_emb, pose_face_emb], dim=1)  # (B, 2, H)
        other_for_right = torch.stack([left_emb, pose_face_emb], dim=1)
        other_for_pf = torch.stack([left_emb, right_emb], dim=1)
        
        left_fused = self.left_fusion(left_emb, other_for_left)
        right_fused = self.right_fusion(right_emb, other_for_right)
        pose_face_fused = self.pose_face_fusion(pose_face_emb, other_for_pf)
        
        # Concatenate and classify
        combined = torch.cat([left_fused, right_fused, pose_face_fused], dim=1)
        combined = self.final_norm(combined)
        
        return self.classifier(combined)

In [ ]:
class FocalLoss(nn.Module):
    """Focal Loss for handling class imbalance"""
    def __init__(self, alpha=0.25, gamma=2.0, label_smoothing=0.1):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.label_smoothing = label_smoothing
    
    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none', 
                                  label_smoothing=self.label_smoothing)
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * ce_loss
        return focal_loss.mean()

In [ ]:
def mixup_data(left, right, pose_face, labels, alpha=0.2):
    """Mixup augmentation"""
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1
    
    batch_size = left.size(0)
    index = torch.randperm(batch_size, device=left.device)
    
    mixed_left = lam * left + (1 - lam) * left[index]
    mixed_right = lam * right + (1 - lam) * right[index]
    mixed_pf = lam * pose_face + (1 - lam) * pose_face[index]
    
    return mixed_left, mixed_right, mixed_pf, labels, labels[index], lam


def mixup_criterion(criterion, pred, y_a, y_b, lam):
    """Mixup loss"""
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

In [ ]:
# ============== DATA LOADING ==============

def load_all_data():
    """Load and preprocess all training data"""
    train_df = pd.read_csv(TRAIN_FILE)
    train_df['sign'] = train_df['sign'].map(SIGN2INDEX)
    
    # Split data
    train_split, val_split = train_test_split(
        train_df, test_size=0.1, stratify=train_df['sign'], random_state=42
    )
    
    print(f"Loading {len(train_split)} training samples...")
    train_data = []
    for path in tqdm(train_split['path'].tolist()):
        try:
            train_data.append(process_video(path))
        except Exception as e:
            print(f"Error loading {path}: {e}")
            # Add dummy data
            train_data.append((
                np.zeros((10, 93), dtype=np.float32),
                np.zeros((10, 93), dtype=np.float32),
                np.zeros((10, 404), dtype=np.float32)
            ))
    
    print(f"Loading {len(val_split)} validation samples...")
    val_data = []
    for path in tqdm(val_split['path'].tolist()):
        try:
            val_data.append(process_video(path))
        except Exception as e:
            print(f"Error loading {path}: {e}")
            val_data.append((
                np.zeros((10, 93), dtype=np.float32),
                np.zeros((10, 93), dtype=np.float32),
                np.zeros((10, 404), dtype=np.float32)
            ))
    
    return train_data, train_split['sign'].to_numpy(), val_data, val_split['sign'].to_numpy()

In [ ]:
# Load data
train_data, train_labels, val_data, val_labels = load_all_data()

# Get feature dimensions from first sample
LEFT_DIM = train_data[0][0].shape[1]
RIGHT_DIM = train_data[0][1].shape[1]
POSE_FACE_DIM = train_data[0][2].shape[1]

print(f"Feature dimensions: Left={LEFT_DIM}, Right={RIGHT_DIM}, PoseFace={POSE_FACE_DIM}")

In [ ]:
# Create datasets and dataloaders
MAX_FRAMES = 128
BATCH_SIZE = 64

train_dataset = ASLMultiStreamDataset(train_data, train_labels, MAX_FRAMES, augment=True)
val_dataset = ASLMultiStreamDataset(val_data, val_labels, MAX_FRAMES, augment=False)

# Get lengths for bucket sampling
train_lengths = [min(d[0].shape[0], MAX_FRAMES) for d in train_data]
val_lengths = [min(d[0].shape[0], MAX_FRAMES) for d in val_data]

train_sampler = BucketBatchSampler(train_lengths, BATCH_SIZE)
val_sampler = BucketBatchSampler(val_lengths, BATCH_SIZE)

train_loader = DataLoader(train_dataset, batch_sampler=train_sampler, collate_fn=collate_multistream)
val_loader = DataLoader(val_dataset, batch_sampler=val_sampler, collate_fn=collate_multistream)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

In [ ]:
# ============== MODEL SETUP ==============

model = MultiStreamASLModel(
    left_dim=LEFT_DIM,
    right_dim=RIGHT_DIM,
    pose_face_dim=POSE_FACE_DIM,
    num_classes=NUM_CLASSES,
    hidden_dim=192,
    tcn_layers=4,
    transformer_layers=3,
    n_heads=6,
    dropout=0.15
).to(device)

# Count parameters
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model parameters: {num_params:,}")

In [ ]:
# Training setup
criterion = FocalLoss(alpha=0.25, gamma=2.0, label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)
scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2, eta_min=1e-6)
scaler = GradScaler(enabled=use_amp)

EPOCHS = 100
MIXUP_ALPHA = 0.2
PATIENCE = 20

In [ ]:
def train_epoch(loader, use_mixup=True):
    model.train()
    total_loss = 0
    
    for left, right, pose_face, mask, labels in loader:
        left = left.to(device)
        right = right.to(device)
        pose_face = pose_face.to(device)
        mask = mask.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        
        with autocast(enabled=use_amp, device_type=device.type):
            if use_mixup and np.random.random() > 0.5:
                left, right, pose_face, labels_a, labels_b, lam = mixup_data(
                    left, right, pose_face, labels, MIXUP_ALPHA
                )
                logits = model(left, right, pose_face, mask)
                loss = mixup_criterion(criterion, logits, labels_a, labels_b, lam)
            else:
                logits = model(left, right, pose_face, mask)
                loss = criterion(logits, labels)
        
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        
        total_loss += loss.item()
    
    return total_loss / len(loader)


@torch.no_grad()
def validate_epoch(loader):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    
    for left, right, pose_face, mask, labels in loader:
        left = left.to(device)
        right = right.to(device)
        pose_face = pose_face.to(device)
        mask = mask.to(device)
        labels = labels.to(device)
        
        with autocast(enabled=use_amp, device_type=device.type):
            logits = model(left, right, pose_face, mask)
            loss = criterion(logits, labels)
        
        total_loss += loss.item()
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    
    return total_loss / len(loader), correct / total

In [ ]:
# ============== TRAINING LOOP ==============

best_val_acc = 0
patience_counter = 0

for epoch in range(EPOCHS):
    train_loss = train_epoch(train_loader, use_mixup=True)
    val_loss, val_acc = validate_epoch(val_loader)
    
    scheduler.step()
    
    print(f"Epoch {epoch:3d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | LR: {optimizer.param_groups[0]['lr']:.2e}")
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': val_acc,
            'val_loss': val_loss,
        }, 'best_model.pth')
        print(f"  -> New best model saved! Acc: {val_acc:.4f}")
    else:
        patience_counter += 1
    
    if patience_counter >= PATIENCE:
        print(f"Early stopping at epoch {epoch}")
        break

print(f"\nBest validation accuracy: {best_val_acc:.4f}")

In [ ]:
# Load best model and final evaluation
checkpoint = torch.load('best_model.pth')
model.load_state_dict(checkpoint['model_state_dict'])

val_loss, val_acc = validate_epoch(val_loader)
print(f"Final validation accuracy: {val_acc:.4f}")